# Baseline U-Net34 — ISIC 2018 trên Kaggle TPU v5e-8

Notebook tự chứa toàn bộ pipeline: Kaggle input → cache uint8 tạm → 3-fold → LR Finder trên train subset → train 128/256 → BestDice → official validation/test → ensemble.

Thiết lập Kaggle trước khi chạy: **Accelerator = TPU v5e-8**, attach dataset **tinlngnguyn/skin-lesion-segmentation-hus**, bật Internet cho lần đầu tải ResNet34 ImageNet. Dùng nguyên bộ torch/torchvision/torch_xla của TPU image; không pip-upgrade riêng lẻ.

> Chạy **Run All đúng một lần**. Nếu đổi accelerator, batch hoặc chạy lại sau lỗi XLA, hãy Restart Session trước.


In [ ]:
#!/usr/bin/env python3
"""Baseline U-Net34 cho Kaggle TPU v5e-8 (PyTorch/XLA SPMD).

Chạy trong Kaggle Notebook đã chọn Accelerator = TPU v5e-8:

    !python /kaggle/working/train_baseline_unet34_tpu_v5e8.py

Script tự chứa toàn bộ pipeline: đọc dataset Kaggle, kiểm tra pairing, tạo cache
uint8 tạm thời, chia 3 fold, LR Finder chỉ trên train subset, train 128 ->
256 (frozen -> unfrozen), chọn BestDice trên fold holdout sạch và cuối cùng
đánh giá official validation/test cùng ensemble 3 fold.

Không cài lại torch/torchvision/torch_xla trong runtime Kaggle: ba package này
phải là bộ phiên bản tương thích do Kaggle TPU image cung cấp.
"""

from __future__ import annotations

import csv
import gc
import hashlib
import json
import math
import os
import random
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any, Iterable, Iterator, Mapping, Sequence

# Các biến XLA phải được đặt trước khi import torch_xla.
os.environ.setdefault("PJRT_DEVICE", "TPU")
os.environ.setdefault("XLA_NO_SPECIAL_SCALARS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import albumentations as A
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torch_xla
import torch_xla.debug.metrics as met
import torch_xla.distributed.parallel_loader as pl
import torch_xla.distributed.spmd as xs
import torch_xla.runtime as xr
from albumentations.pytorch import ToTensorV2
from PIL import Image
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset, Sampler, get_worker_info
from torchvision import models




In [ ]:
# ---------------------------------------------------------------------------
# Cấu hình
# ---------------------------------------------------------------------------

DATA_ROOT = Path("/kaggle/input/skin-lesion-segmentation-hus")
OUTPUT_ROOT = Path("/kaggle/working/unet34_baseline_tpu_v5e8")
RUN_NAME = "full_spmd_v2"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"

# Cache này là dữ liệu dẫn xuất tạm thời, không nằm trong artifact đầu ra.
DATA_CACHE_DIR = Path("/tmp/isic_unet34_uint8_cache_v1")
XLA_CACHE_DIR = Path("/tmp/isic_unet34_xla_cache_v1")

SEED = 42
N_FOLDS = 3
SIZES = (128, 256)
FROZEN_EPOCHS = 30
UNFROZEN_EPOCHS = 30

# Global batch được XLA shard đều qua 8 chip: 32/16 mẫu mỗi chip.
# Nếu runtime báo XLA OOM, restart session rồi giảm thành {128: 128, 256: 64}.
TRAIN_BATCH_BY_SIZE = {128: 256, 256: 128}
EVAL_BATCH_BY_SIZE = {128: 128, 256: 128}

TRAIN_NUM_WORKERS = 4
EVAL_NUM_WORKERS = 2
PREFETCH_FACTOR = 2
XLA_LOADER_PREFETCH = 4
XLA_DEVICE_PREFETCH = 2
EXPECTED_TPU_DEVICES = 8

LRF_ITERS = 50
LR_MIN = 1e-6
LR_MAX = 1.0
LRF_BETA = 0.9
LR_SELECTION_FACTOR = 0.1
MAX_LR_FROZEN = 3e-3
MAX_LR_UNFROZEN = 1e-3
CUT_FRAC = 0.1
STLR_RATIO = 32.0

WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
PRED_THRESHOLD = 0.5
JACCARD_CUT = 0.65
SAVE_LAST_EVERY = 5
PRETRAINED = True
RESUME = True
PROTOCOL_VERSION = 9
MODEL_REVISION = "unet34_no_registered_alias_v2"

# Chỉ dùng để xác nhận pipeline trên TPU; mặc định vẫn train đầy đủ.
FAST_DEV_RUN = os.getenv("FAST_DEV_RUN", "0") == "1"
if FAST_DEV_RUN:
    RUN_NAME = "fast_dev_spmd_v2"
    RUN_DIR = OUTPUT_ROOT / RUN_NAME
    CHECKPOINT_DIR = RUN_DIR / "checkpoints"
    LOG_DIR = RUN_DIR / "logs"
    SIZES = (128,)
    FROZEN_EPOCHS = 1
    UNFROZEN_EPOCHS = 1
    LRF_ITERS = 8

SPLIT_DIRECTORIES = {
    "train": (
        "ISIC2018_Task1-2_Training_Input",
        "ISIC2018_Task1_Training_GroundTruth",
    ),
    "validation": (
        "ISIC2018_Task1-2_Validation_Input",
        "ISIC2018_Task1_Validation_GroundTruth",
    ),
    "test": (
        "ISIC2018_Task1-2_Test_Input",
        "ISIC2018_Task1_Test_GroundTruth",
    ),
}
EXPECTED_COUNTS = {"train": 2594, "validation": 100, "test": 1000}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
CACHE_VERSION = 1




In [ ]:
# ---------------------------------------------------------------------------
# Tiện ích chung và khám phá dữ liệu
# ---------------------------------------------------------------------------

def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def deterministic_subset(records: Sequence[dict[str, str]], count: int, seed: int):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))[: min(count, len(records))]
    return [records[int(index)] for index in indices]


def find_populated_directory(name: str, suffixes: set[str]) -> Path:
    outer = DATA_ROOT / name
    seen: set[Path] = set()

    def populated(candidates: Iterable[Path]) -> Path | None:
        for candidate in candidates:
            candidate = candidate.resolve()
            if candidate in seen or not candidate.is_dir():
                continue
            seen.add(candidate)
            if any(
                path.is_file() and path.suffix.lower() in suffixes
                for path in candidate.iterdir()
            ):
                return candidate
        return None

    direct = populated((outer / name, outer))
    if direct is not None:
        return direct
    if DATA_ROOT.is_dir():
        nested = populated(path for path in DATA_ROOT.rglob(name) if path.is_dir())
        if nested is not None:
            return nested
    raise FileNotFoundError(f"Không tìm thấy thư mục có dữ liệu: {name} dưới {DATA_ROOT}")


def discover_records(split: str) -> list[dict[str, str]]:
    image_name, mask_name = SPLIT_DIRECTORIES[split]
    image_dir = find_populated_directory(image_name, IMAGE_SUFFIXES)
    mask_dir = find_populated_directory(mask_name, {".png"})
    image_map = {
        path.stem: path
        for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    }
    mask_map = {
        path.stem.removesuffix("_segmentation"): path
        for path in mask_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() == ".png"
        and path.stem.endswith("_segmentation")
    }
    missing_masks = sorted(set(image_map) - set(mask_map))
    orphan_masks = sorted(set(mask_map) - set(image_map))
    if missing_masks or orphan_masks:
        raise RuntimeError(
            f"{split}: missing_masks={missing_masks[:5]}, "
            f"orphan_masks={orphan_masks[:5]}"
        )
    records = [
        {
            "image_id": image_id,
            "image_path": str(image_map[image_id]),
            "mask_path": str(mask_map[image_id]),
        }
        for image_id in sorted(image_map)
    ]
    expected = EXPECTED_COUNTS[split]
    if len(records) != expected:
        raise RuntimeError(f"{split}: cần {expected} cặp, tìm thấy {len(records)}")
    return records


def make_folds(
    records: Sequence[dict[str, str]], n_splits: int = N_FOLDS, seed: int = SEED
) -> list[tuple[list[dict[str, str]], list[dict[str, str]]]]:
    permutation = torch.randperm(
        len(records), generator=torch.Generator().manual_seed(seed)
    ).tolist()
    fold_sizes = [len(records) // n_splits] * n_splits
    for index in range(len(records) % n_splits):
        fold_sizes[index] += 1
    all_indices = set(range(len(records)))
    folds = []
    offset = 0
    for fold_size in fold_sizes:
        holdout_indices = set(permutation[offset : offset + fold_size])
        train_indices = all_indices - holdout_indices
        if train_indices & holdout_indices or train_indices | holdout_indices != all_indices:
            raise AssertionError("Fold split không tạo thành partition hợp lệ")
        folds.append(
            (
                [records[index] for index in sorted(train_indices)],
                [records[index] for index in sorted(holdout_indices)],
            )
        )
        offset += fold_size
    return folds


def ids_digest(records: Sequence[Mapping[str, str]]) -> str:
    joined = "\n".join(record["image_id"] for record in records)
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()




In [ ]:
# ---------------------------------------------------------------------------
# Cache uint8 dạng memmap: khoảng 1.06 GiB, không nhân theo 8 chip
# ---------------------------------------------------------------------------

def gray_world(image: np.ndarray) -> np.ndarray:
    image_float = image.astype(np.float32)
    channel_means = image_float.mean(axis=(0, 1))
    gray_mean = float(channel_means.mean())
    scale = gray_mean / np.maximum(channel_means, 1e-6)
    corrected = image_float * scale.reshape(1, 1, 3)
    return np.clip(corrected, 0, 255).astype(np.uint8)


def cache_signature(records: Sequence[Mapping[str, str]], size: int) -> str:
    digest = hashlib.sha256()
    digest.update(f"v{CACHE_VERSION}|{size}".encode("ascii"))
    for record in records:
        digest.update(record["image_id"].encode("utf-8"))
        digest.update(record["image_path"].encode("utf-8"))
        digest.update(record["mask_path"].encode("utf-8"))
    return digest.hexdigest()


def _decode_and_resize(record: Mapping[str, str], size: int):
    with Image.open(record["image_path"]) as source:
        image = np.asarray(source.convert("RGB"), dtype=np.uint8).copy()
    with Image.open(record["mask_path"]) as source:
        mask = np.asarray(source.convert("L"), dtype=np.uint8).copy()
    if image.shape[:2] != mask.shape:
        raise ValueError(
            f"Kích thước ảnh/mask lệch cho {record['image_id']}: "
            f"{image.shape[:2]} và {mask.shape}"
        )
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    image = gray_world(image)
    mask = cv2.resize(
        (mask > 127).astype(np.uint8),
        (size, size),
        interpolation=cv2.INTER_NEAREST,
    )
    return image, mask


def build_or_reuse_cache(
    records: Sequence[dict[str, str]], size: int
) -> dict[str, Any]:
    DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    prefix = DATA_CACHE_DIR / f"cache_{size}px"
    image_path = prefix.with_name(prefix.name + "_images.npy")
    mask_path = prefix.with_name(prefix.name + "_masks.npy")
    meta_path = prefix.with_name(prefix.name + "_meta.json")
    signature = cache_signature(records, size)
    expected_meta = {
        "version": CACHE_VERSION,
        "size": int(size),
        "count": len(records),
        "signature": signature,
        "ids": [record["image_id"] for record in records],
    }

    reusable = False
    if image_path.is_file() and mask_path.is_file() and meta_path.is_file():
        try:
            current_meta = json.loads(meta_path.read_text(encoding="utf-8"))
            images = np.load(image_path, mmap_mode="r")
            masks = np.load(mask_path, mmap_mode="r")
            reusable = (
                current_meta == expected_meta
                and images.shape == (len(records), size, size, 3)
                and masks.shape == (len(records), size, size)
                and images.dtype == np.uint8
                and masks.dtype == np.uint8
            )
            del images, masks
        except (OSError, ValueError, json.JSONDecodeError):
            reusable = False

    if not reusable:
        image_tmp = image_path.with_suffix(".npy.tmp")
        mask_tmp = mask_path.with_suffix(".npy.tmp")
        meta_tmp = meta_path.with_suffix(".json.tmp")
        images = np.lib.format.open_memmap(
            image_tmp,
            mode="w+",
            dtype=np.uint8,
            shape=(len(records), size, size, 3),
        )
        masks = np.lib.format.open_memmap(
            mask_tmp,
            mode="w+",
            dtype=np.uint8,
            shape=(len(records), size, size),
        )
        workers = min(4, max(1, os.cpu_count() or 1))
        print(f"[cache] tạo {size}px: {len(records)} cặp, {workers} decode threads")
        with ThreadPoolExecutor(max_workers=workers) as executor:
            chunk_size = workers * 2
            for start in range(0, len(records), chunk_size):
                stop = min(start + chunk_size, len(records))
                decoded = executor.map(
                    lambda record: _decode_and_resize(record, size),
                    records[start:stop],
                )
                for index, (image, mask) in enumerate(decoded, start=start):
                    images[index] = image
                    masks[index] = mask
                if stop == len(records) or stop % 256 < chunk_size:
                    print(f"[cache] {size}px: {stop}/{len(records)}")
        images.flush()
        masks.flush()
        del images, masks
        meta_tmp.write_text(
            json.dumps(expected_meta, ensure_ascii=False), encoding="utf-8"
        )
        os.replace(image_tmp, image_path)
        os.replace(mask_tmp, mask_path)
        os.replace(meta_tmp, meta_path)
    else:
        print(f"[cache] dùng lại cache {size}px")

    return {
        "image_path": str(image_path),
        "mask_path": str(mask_path),
        "ids": expected_meta["ids"],
        "size": size,
    }




In [ ]:
# ---------------------------------------------------------------------------
# Dataset, augmentation và batch shape cố định
# ---------------------------------------------------------------------------

def build_transform(train: bool, seed: int) -> A.Compose:
    steps: list[Any] = []
    if train:
        steps.extend(
            [
                A.D4(p=1.0),
                A.Affine(
                    rotate=(-44, 44),
                    scale=(1.0, 1.05),
                    interpolation=cv2.INTER_LINEAR,
                    mask_interpolation=cv2.INTER_NEAREST,
                    p=0.75,
                ),
                A.RandomBrightnessContrast(
                    brightness_limit=0.2, contrast_limit=0.2, p=0.75
                ),
            ]
        )
    steps.append(ToTensorV2())
    return A.Compose(steps, seed=int(seed))


class CachedSegmentationDataset(Dataset):
    def __init__(
        self,
        records: Sequence[dict[str, str]],
        cache_spec: Mapping[str, Any],
        train: bool,
        seed: int,
    ) -> None:
        if not records:
            raise ValueError("Dataset rỗng")
        self.records = list(records)
        self.image_path = str(cache_spec["image_path"])
        self.mask_path = str(cache_spec["mask_path"])
        id_to_index = {
            image_id: index for index, image_id in enumerate(cache_spec["ids"])
        }
        try:
            self.cache_indices = [
                id_to_index[record["image_id"]] for record in self.records
            ]
        except KeyError as error:
            raise RuntimeError(f"Cache thiếu image id: {error.args[0]}") from error
        self.train = bool(train)
        self.transform = build_transform(self.train, seed)
        self._open_memmaps()

    def _open_memmaps(self) -> None:
        self.images = np.load(self.image_path, mmap_mode="r")
        self.masks = np.load(self.mask_path, mmap_mode="r")

    def __getstate__(self):
        state = self.__dict__.copy()
        state["images"] = None
        state["masks"] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._open_memmaps()

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int | tuple[int, int]) -> dict[str, torch.Tensor]:
        augmentation_seed = None
        if isinstance(index, tuple):
            index, augmentation_seed = index
        cache_index = self.cache_indices[index]
        image = np.asarray(self.images[cache_index]).copy()
        mask = np.asarray(self.masks[cache_index]).copy()
        if self.train and augmentation_seed is not None:
            # Giữ augmentation giống nhau khi resume dù worker RNG đã ở trạng thái khác.
            self.transform.set_random_seed(int(augmentation_seed))
        transformed = self.transform(image=image, mask=mask)
        image_tensor = transformed["image"].contiguous()
        mask_tensor = transformed["mask"]
        if mask_tensor.ndim == 2:
            mask_tensor = mask_tensor.unsqueeze(0)
        return {
            "image": image_tensor,
            # Giữ uint8 khi H2D; train/eval mới cast FP32 trên TPU.
            "mask": (mask_tensor > 0).to(torch.uint8).contiguous(),
            "valid": torch.tensor(1.0, dtype=torch.float32),
        }


class PaddedEvalDataset(Dataset):
    def __init__(self, base: CachedSegmentationDataset, batch_size: int) -> None:
        self.base = base
        self.real_length = len(base)
        self.padded_length = (
            math.ceil(self.real_length / int(batch_size)) * int(batch_size)
        )

    def __len__(self) -> int:
        return self.padded_length

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        valid = index < self.real_length
        sample = self.base[index if valid else index % self.real_length]
        if not valid:
            sample["valid"] = torch.tensor(0.0, dtype=torch.float32)
        return sample


class FixedRandomSampler(Sampler[tuple[int, int]]):
    """Shuffle rồi pad tới batch cố định; mỗi epoch đổi các mẫu được lặp."""

    def __init__(self, length: int, batch_size: int, seed: int) -> None:
        self.length = int(length)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.epoch = 0
        self.forced_batches: int | None = None
        if self.length < 1 or self.batch_size < 1:
            raise ValueError("length và batch_size phải dương")

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch)

    def set_forced_batches(self, batches: int | None) -> None:
        self.forced_batches = None if batches is None else int(batches)

    def __len__(self) -> int:
        if self.forced_batches is not None:
            return self.forced_batches * self.batch_size
        return math.ceil(self.length / self.batch_size) * self.batch_size

    def __iter__(self) -> Iterator[tuple[int, int]]:
        target_length = len(self)
        indices: list[int] = []
        cycle = 0
        while len(indices) < target_length:
            generator = torch.Generator().manual_seed(
                self.seed + self.epoch * 1_000_003 + cycle
            )
            indices.extend(torch.randperm(self.length, generator=generator).tolist())
            cycle += 1
        sample_seed_base = self.seed + self.epoch * 10_000_019
        return iter(
            (index, sample_seed_base + position)
            for position, index in enumerate(indices[:target_length])
        )


def seed_worker(worker_id: int) -> None:
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    worker = get_worker_info()
    dataset = None if worker is None else worker.dataset
    while hasattr(dataset, "base"):
        dataset = dataset.base
    transform = getattr(dataset, "transform", None)
    if transform is not None and hasattr(transform, "set_random_seed"):
        transform.set_random_seed(worker_seed)


def make_cpu_loader(
    dataset: Dataset,
    batch_size: int,
    workers: int,
    seed: int,
    sampler: Sampler[Any] | None = None,
    persistent: bool = True,
) -> DataLoader:
    kwargs: dict[str, Any] = {
        "dataset": dataset,
        "batch_size": int(batch_size),
        "sampler": sampler,
        "shuffle": False,
        "drop_last": True,
        "num_workers": int(workers),
        "pin_memory": False,
        "persistent_workers": bool(persistent and workers > 0),
        "worker_init_fn": seed_worker,
        "generator": torch.Generator().manual_seed(int(seed)),
    }
    if workers > 0:
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
        # Notebook định nghĩa Dataset trong __main__, vì vậy spawn không thể import
        # class đó. Kaggle/Linux và ví dụ SPMD chính thức dùng worker kiểu fork.
        kwargs["multiprocessing_context"] = "fork"
    return DataLoader(**kwargs)


def make_device_loader(
    cpu_loader: DataLoader,
    device: torch.device,
    input_sharding: Mapping[str, xs.ShardingSpec],
) -> pl.MpDeviceLoader:
    return pl.MpDeviceLoader(
        cpu_loader,
        device,
        input_sharding=input_sharding,
        loader_prefetch_size=XLA_LOADER_PREFETCH,
        device_prefetch_size=XLA_DEVICE_PREFETCH,
        host_to_device_transfer_threads=1,
    )




In [ ]:
# ---------------------------------------------------------------------------
# U-Net34
# ---------------------------------------------------------------------------

class UpBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int) -> None:
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch // 2, 2, stride=2)
        self.skip_projection = nn.Conv2d(skip_ch, out_ch // 2, 1)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        skip = self.skip_projection(skip)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
        return self.bn(F.relu(torch.cat([x, skip], dim=1)))


class UNet34(nn.Module):
    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        try:
            backbone = models.resnet34(weights=weights)
        except Exception as error:
            raise RuntimeError(
                "Không tải được ResNet34 ImageNet weights. Bật Internet trong Kaggle "
                "cho lần tải đầu hoặc attach sẵn checkpoint; không tự fallback random."
            ) from error
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 256)
        self.up2 = UpBlock(256, 64, 256)
        self.up1 = UpBlock(256, 64, 256)
        self.up0 = UpBlock(256, 3, 16)
        self.head = nn.Conv2d(16, 1, 1)

    @property
    def first_layer_group(self) -> tuple[nn.Module, nn.Module, nn.Module]:
        # Property tránh đăng ký lại cùng module và sinh key state_dict trùng.
        return self.stem, self.layer1, self.layer2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        bottleneck = self.layer4(s3)
        decoded = self.up4(bottleneck, s3)
        decoded = self.up3(decoded, s2)
        decoded = self.up2(decoded, s1)
        decoded = self.up1(decoded, s0)
        decoded = self.up0(decoded, x)
        return self.head(decoded)


def freeze_first_layer_group(model: UNet34) -> None:
    """Freeze phần sớm của encoder nhưng vẫn cho BatchNorm thích nghi.

    Đây là đúng contract của baseline gốc, không có nghĩa là freeze toàn encoder.
    """
    for parameter in model.parameters():
        parameter.requires_grad = True
    for module in model.first_layer_group:
        for child in module.modules():
            for parameter in child.parameters(recurse=False):
                parameter.requires_grad = isinstance(child, nn.BatchNorm2d)


def unfreeze_with_frozen_bn(model: UNet34) -> None:
    for parameter in model.parameters():
        parameter.requires_grad = True
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            for parameter in module.parameters():
                parameter.requires_grad = False


def set_train_mode(model: UNet34, bn_frozen: bool) -> None:
    model.train()
    if bn_frozen:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()


def trainable_parameters(model: nn.Module) -> list[nn.Parameter]:
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not parameters:
        raise RuntimeError("Không có parameter trainable")
    return parameters




In [ ]:
# ---------------------------------------------------------------------------
# Metric
# ---------------------------------------------------------------------------

def per_image_scores(
    probability: torch.Tensor,
    target: torch.Tensor,
    threshold: float = PRED_THRESHOLD,
):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    prediction_sum = prediction.sum(dims)
    target_sum = target.sum(dims)
    union = prediction_sum + target_sum - intersection
    jaccard = torch.where(
        union > 0, intersection / union.clamp(min=1), torch.ones_like(union)
    )
    dice = torch.where(
        prediction_sum + target_sum > 0,
        2 * intersection / (prediction_sum + target_sum).clamp(min=1),
        torch.ones_like(union),
    )
    sensitivity = torch.where(
        target_sum > 0,
        intersection / target_sum.clamp(min=1),
        torch.ones_like(target_sum),
    )
    return jaccard, dice, sensitivity


def surface(mask: np.ndarray) -> np.ndarray:
    eroded = ndimage.binary_erosion(mask, border_value=0)
    return np.logical_xor(mask, eroded)


def hd95_single(prediction: np.ndarray, target: np.ndarray) -> float:
    prediction = prediction.astype(bool)
    target = target.astype(bool)
    height, width = target.shape
    if not prediction.any() and not target.any():
        return 0.0
    if not prediction.any() or not target.any():
        return float(math.hypot(height, width))
    prediction_surface = surface(prediction)
    target_surface = surface(target)
    distance_to_target = ndimage.distance_transform_edt(~target_surface)
    distance_to_prediction = ndimage.distance_transform_edt(~prediction_surface)
    distances = np.concatenate(
        [
            distance_to_target[prediction_surface],
            distance_to_prediction[target_surface],
        ]
    )
    return float(np.percentile(distances, 95)) if distances.size else 0.0


def prepare_inputs(
    images: torch.Tensor, mean: torch.Tensor, std: torch.Tensor
) -> torch.Tensor:
    # H2D giữ uint8 để giảm băng thông; convert/normalize chạy trong graph XLA.
    return images.float().div(255.0).sub(mean).div(std)


@torch.inference_mode()
def ensemble_probability(
    models_list: Sequence[UNet34], inputs: torch.Tensor
) -> torch.Tensor:
    total: torch.Tensor | None = None
    for model in models_list:
        model.eval()
        with torch.autocast("xla", dtype=torch.bfloat16):
            probability = torch.sigmoid(model(inputs).float())
        total = probability if total is None else total + probability
    if total is None:
        raise ValueError("Danh sách model rỗng")
    return total / len(models_list)


@torch.inference_mode()
def evaluate(
    models_list: Sequence[UNet34],
    cpu_loader: DataLoader,
    device: torch.device,
    input_sharding: Mapping[str, xs.ShardingSpec],
    mean: torch.Tensor,
    std: torch.Tensor,
    include_hd95: bool = False,
) -> dict[str, float]:
    # Trong SPMD đây là tensor logic toàn cục; XLA chèn collective khi reduce.
    sums = torch.zeros(5, device=device, dtype=torch.float32)
    hd95_sum = 0.0
    device_loader = make_device_loader(cpu_loader, device, input_sharding)
    for batch in device_loader:
        inputs = prepare_inputs(batch["image"], mean, std)
        targets = batch["mask"].float()
        valid = batch["valid"].float()
        probability = ensemble_probability(models_list, inputs)
        jaccard, dice, sensitivity = per_image_scores(probability, targets)
        sums = sums + torch.stack(
            [
                (jaccard * valid).sum(),
                (jaccard * (jaccard >= JACCARD_CUT) * valid).sum(),
                (dice * valid).sum(),
                (sensitivity * valid).sum(),
                valid.sum(),
            ]
        )
        if include_hd95:
            probability_cpu = probability.cpu().numpy()[:, 0]
            targets_cpu = targets.cpu().numpy()[:, 0] > 0.5
            valid_cpu = valid.cpu().numpy() > 0.5
            hd95_sum += sum(
                hd95_single(prediction > PRED_THRESHOLD, target)
                for prediction, target, keep in zip(
                    probability_cpu, targets_cpu, valid_cpu
                )
                if keep
            )
    torch_xla.sync(wait=True)
    values = sums.cpu().numpy().astype(np.float64)
    count = float(values[4])
    if not np.isfinite(values).all() or count <= 0:
        raise FloatingPointError(f"Metric không hợp lệ: {values.tolist()}")
    metrics = {
        "jaccard": float(values[0] / count),
        "threshold_jaccard": float(values[1] / count),
        "dice": float(values[2] / count),
        "sensitivity": float(values[3] / count),
    }
    if include_hd95:
        metrics["hd95"] = float(hd95_sum / count)
    return metrics




In [ ]:
# ---------------------------------------------------------------------------
# Checkpoint, LR Finder và STLR không recompile theo từng scalar LR
# ---------------------------------------------------------------------------

def atomic_torch_save(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def trusted_torch_load(path: Path, map_location: str | torch.device = "cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def tree_to_cpu(value: Any) -> Any:
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: tree_to_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [tree_to_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(tree_to_cpu(item) for item in value)
    return value


def cpu_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:
    torch_xla.sync(wait=True)
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def load_model_state(model: nn.Module, state: Mapping[str, torch.Tensor]) -> None:
    while state and all(key.startswith("module.") for key in state):
        state = {key.removeprefix("module."): value for key, value in state.items()}
    model.load_state_dict(state)


def tensor_lr_optimizer(
    model: nn.Module, learning_rate: torch.Tensor
) -> torch.optim.Adam:
    # XLA được PyTorch liệt kê là capturable device. Tensor LR cho phép LR Finder
    # và STLR thay đổi giá trị mà không tạo graph mới cho mỗi Python float.
    return torch.optim.Adam(
        trainable_parameters(model),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
        foreach=False,
        fused=False,
        capturable=True,
    )


def stlr_value(
    max_lr: float,
    step: int,
    total_steps: int,
    cut_frac: float = CUT_FRAC,
    ratio: float = STLR_RATIO,
) -> float:
    cut = max(1, int(total_steps * cut_frac))
    if step < cut:
        progress = step / cut
    else:
        progress = max(0.0, 1.0 - (step - cut) / max(1, total_steps - cut))
    return max_lr * (1.0 + progress * (ratio - 1.0)) / ratio


def update_stlr_tensor(
    learning_rate: torch.Tensor,
    step_tensor: torch.Tensor,
    max_lr: float,
    total_steps: int,
) -> None:
    cut = max(1, int(total_steps * CUT_FRAC))
    warm_progress = step_tensor / float(cut)
    cool_progress = 1.0 - (step_tensor - float(cut)) / float(
        max(1, total_steps - cut)
    )
    progress = torch.where(step_tensor < cut, warm_progress, cool_progress)
    progress = progress.clamp(0.0, 1.0)
    learning_rate.copy_(
        max_lr * (1.0 + progress * (STLR_RATIO - 1.0)) / STLR_RATIO
    )


def lr_find(
    model: UNet34,
    train_loader: DataLoader,
    train_sampler: FixedRandomSampler,
    device: torch.device,
    input_sharding: Mapping[str, xs.ShardingSpec],
    mean: torch.Tensor,
    std: torch.Tensor,
    stage_seed: int,
    size: int,
    bn_frozen: bool,
) -> tuple[float, list[float], list[float]]:
    set_seed(stage_seed)
    original_state = cpu_state_dict(model)
    learning_rate = torch.tensor(LR_MIN, device=device, dtype=torch.float32)
    multiplier = float((LR_MAX / LR_MIN) ** (1.0 / max(1, LRF_ITERS - 1)))
    optimizer = tensor_lr_optimizer(model, learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    train_sampler.set_epoch(100_000 + stage_seed)
    train_sampler.set_forced_batches(LRF_ITERS)
    device_loader = make_device_loader(train_loader, device, input_sharding)
    loss_tensors: list[torch.Tensor] = []

    set_train_mode(model, bn_frozen=bn_frozen)
    for batch in device_loader:
        inputs = prepare_inputs(batch["image"], mean, std)
        targets = batch["mask"].float()
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("xla", dtype=torch.bfloat16):
            loss = criterion(model(inputs).float(), targets)
        loss.backward()
        if GRAD_CLIP_NORM > 0:
            nn.utils.clip_grad_norm_(trainable_parameters(model), GRAD_CLIP_NORM)
        optimizer.step()
        loss_tensors.append(loss.detach())
        learning_rate.mul_(multiplier)

    torch_xla.sync(wait=True)
    raw_losses = torch.stack(loss_tensors).cpu().numpy().astype(np.float64)
    learning_rates = LR_MIN * multiplier ** np.arange(len(raw_losses))
    load_model_state(model, original_state)
    train_sampler.set_forced_batches(None)
    del optimizer, loss_tensors, original_state
    gc.collect()

    smoothed_losses: list[float] = []
    usable_lrs: list[float] = []
    moving_average = 0.0
    best_loss = float("inf")
    for iteration, (current_lr, raw_loss) in enumerate(
        zip(learning_rates, raw_losses)
    ):
        if not np.isfinite(raw_loss):
            break
        moving_average = LRF_BETA * moving_average + (1.0 - LRF_BETA) * raw_loss
        smoothed = moving_average / (1.0 - LRF_BETA ** (iteration + 1))
        if iteration > 5 and smoothed > 4.0 * best_loss:
            break
        best_loss = min(best_loss, smoothed)
        usable_lrs.append(float(current_lr))
        smoothed_losses.append(float(smoothed))
    if len(usable_lrs) < 3:
        raise RuntimeError("LR Finder không có đủ điểm hữu hạn để chọn learning rate")
    lr_array = np.asarray(usable_lrs, dtype=np.float64)
    loss_array = np.asarray(smoothed_losses, dtype=np.float64)
    gradient = np.gradient(loss_array, np.log10(lr_array))
    steepest_lr = float(lr_array[int(np.argmin(gradient))])
    min_loss_lr = float(lr_array[int(np.argmin(loss_array))])
    lr_cap = MAX_LR_UNFROZEN if bn_frozen else MAX_LR_FROZEN
    selected_lr = max(
        LR_MIN,
        min(steepest_lr * LR_SELECTION_FACTOR, min_loss_lr / 10.0, lr_cap),
    )
    print(
        f"LR Finder {size}px: steepest={steepest_lr:.2e}, "
        f"min-loss={min_loss_lr:.2e}, selected={selected_lr:.2e}"
    )
    set_seed(stage_seed)
    return selected_lr, usable_lrs, smoothed_losses


def write_rows(path: Path, rows: Sequence[Mapping[str, Any]]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def make_stage_loaders(
    train_records: Sequence[dict[str, str]],
    holdout_records: Sequence[dict[str, str]],
    cache_spec: Mapping[str, Any],
    size: int,
    seed: int,
):
    train_batch = TRAIN_BATCH_BY_SIZE[size]
    eval_batch = EVAL_BATCH_BY_SIZE[size]
    train_dataset = CachedSegmentationDataset(
        train_records, cache_spec, train=True, seed=seed
    )
    train_sampler = FixedRandomSampler(len(train_dataset), train_batch, seed)
    train_loader = make_cpu_loader(
        train_dataset,
        batch_size=train_batch,
        workers=TRAIN_NUM_WORKERS,
        seed=seed,
        sampler=train_sampler,
        persistent=True,
    )
    holdout_base = CachedSegmentationDataset(
        holdout_records, cache_spec, train=False, seed=seed
    )
    holdout_dataset = PaddedEvalDataset(holdout_base, eval_batch)
    holdout_loader = make_cpu_loader(
        holdout_dataset,
        batch_size=eval_batch,
        workers=EVAL_NUM_WORKERS,
        seed=seed,
        sampler=None,
        persistent=True,
    )
    return train_loader, train_sampler, holdout_loader


def run_stage(
    model: UNet34,
    train_records: Sequence[dict[str, str]],
    holdout_records: Sequence[dict[str, str]],
    cache_spec: Mapping[str, Any],
    size: int,
    epochs: int,
    bn_frozen: bool,
    fold_number: int,
    stage_name: str,
    stage_seed: int,
    device: torch.device,
    input_sharding: Mapping[str, xs.ShardingSpec],
    mean: torch.Tensor,
    std: torch.Tensor,
    run_config: Mapping[str, Any],
) -> dict[str, Any]:
    stage_dir = CHECKPOINT_DIR / f"fold_{fold_number}" / f"{size}px_{stage_name}"
    best_path = stage_dir / "best.pt"
    last_path = stage_dir / "last.pt"
    stage_contract = {
        "protocol_version": PROTOCOL_VERSION,
        "fold": fold_number,
        "size": size,
        "stage": stage_name,
        "epochs": epochs,
        "stage_seed": stage_seed,
        "bn_frozen": bn_frozen,
        "train_ids_sha256": ids_digest(train_records),
        "holdout_ids_sha256": ids_digest(holdout_records),
        "train_batch": TRAIN_BATCH_BY_SIZE[size],
        "eval_batch": EVAL_BATCH_BY_SIZE[size],
        "lr_finder_iterations": LRF_ITERS,
        "optimizer": "Adam(capturable tensor LR, foreach=False)",
        "precision": "BF16 autocast, FP32 state",
        "run_config": dict(run_config),
    }

    train_loader, train_sampler, holdout_loader = make_stage_loaders(
        train_records, holdout_records, cache_spec, size, stage_seed
    )
    steps_per_epoch = math.ceil(
        len(train_records) / TRAIN_BATCH_BY_SIZE[size]
    )
    if len(train_loader) != steps_per_epoch:
        raise AssertionError(
            f"Sai số step: loader={len(train_loader)}, expected={steps_per_epoch}"
        )

    saved = None
    if RESUME and last_path.is_file():
        saved = trusted_torch_load(last_path)
        if saved.get("contract") != stage_contract:
            raise RuntimeError(
                f"Checkpoint không khớp config/split: {last_path}. "
                "Đổi RUN_NAME hoặc xóa đúng run cũ."
            )
        if saved.get("complete"):
            best_payload = trusted_torch_load(best_path)
            load_model_state(model, best_payload["model_state"])
            print(
                f"[resume] fold={fold_number} {size}px {stage_name} đã hoàn tất"
            )
            del train_loader, holdout_loader
            gc.collect()
            return saved["result"]

    if saved is None:
        selected_lr, lrf_lrs, lrf_losses = lr_find(
            model=model,
            train_loader=train_loader,
            train_sampler=train_sampler,
            device=device,
            input_sharding=input_sharding,
            mean=mean,
            std=std,
            stage_seed=stage_seed,
            size=size,
            bn_frozen=bn_frozen,
        )
        initial_metrics = evaluate(
            [model],
            holdout_loader,
            device,
            input_sharding,
            mean,
            std,
            include_hd95=False,
        )
        best_score = float(initial_metrics["dice"])
        best_metrics = initial_metrics
        best_epoch = 0
        best_state = cpu_state_dict(model)
        history: list[dict[str, Any]] = []
        start_epoch = 0
    else:
        selected_lr = float(saved["selected_lr"])
        lrf_lrs = list(saved["lrf_lrs"])
        lrf_losses = list(saved["lrf_losses"])
        best_score = float(saved["best_score"])
        best_metrics = dict(saved["best_metrics"])
        best_epoch = int(saved["best_epoch"])
        history = list(saved["history"])
        start_epoch = int(saved["next_epoch"])
        load_model_state(model, saved["model_state"])
        best_state = saved.get("best_model_state")
        if best_state is None and best_path.is_file():
            # Tương thích checkpoint v1 nếu người dùng chủ động giữ RUN_NAME cũ.
            best_state = trusted_torch_load(best_path)["model_state"]
        if best_state is None:
            raise RuntimeError(f"Checkpoint resume thiếu BestDice state: {last_path}")
        print(
            f"[resume] fold={fold_number} {size}px {stage_name} "
            f"từ epoch {start_epoch + 1}"
        )

    total_steps = epochs * steps_per_epoch
    completed_steps = start_epoch * steps_per_epoch
    learning_rate = torch.tensor(
        stlr_value(selected_lr, completed_steps, total_steps),
        device=device,
        dtype=torch.float32,
    )
    step_tensor = torch.tensor(
        float(completed_steps), device=device, dtype=torch.float32
    )
    optimizer = tensor_lr_optimizer(model, learning_rate)
    if saved is not None:
        optimizer.load_state_dict(saved["optimizer_state"])
        # load_state_dict có thể thay Tensor LR bằng bản CPU từ checkpoint.
        optimizer.param_groups[0]["lr"] = learning_rate
        saved = None
        gc.collect()
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(start_epoch, epochs):
        train_sampler.set_forced_batches(None)
        train_sampler.set_epoch(epoch)
        set_train_mode(model, bn_frozen=bn_frozen)
        torch_xla.sync(wait=True)
        started = time.perf_counter()
        loss_sum = torch.zeros((), device=device, dtype=torch.float32)

        device_loader = make_device_loader(train_loader, device, input_sharding)
        for batch in device_loader:
            inputs = prepare_inputs(batch["image"], mean, std)
            targets = batch["mask"].float()
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("xla", dtype=torch.bfloat16):
                loss = criterion(model(inputs).float(), targets)
            loss.backward()
            if GRAD_CLIP_NORM > 0:
                nn.utils.clip_grad_norm_(
                    trainable_parameters(model), GRAD_CLIP_NORM
                )
            optimizer.step()
            loss_sum = loss_sum + loss.detach()
            step_tensor.add_(1.0)
            update_stlr_tensor(
                learning_rate, step_tensor, selected_lr, total_steps
            )

        torch_xla.sync(wait=True)
        epoch_seconds = time.perf_counter() - started
        epoch_loss = float((loss_sum / steps_per_epoch).cpu())
        if not np.isfinite(epoch_loss):
            raise FloatingPointError(
                f"NaN/Inf ở fold={fold_number}, size={size}, "
                f"stage={stage_name}, epoch={epoch + 1}"
            )
        metrics = evaluate(
            [model],
            holdout_loader,
            device,
            input_sharding,
            mean,
            std,
            include_hd95=False,
        )
        current_step = (epoch + 1) * steps_per_epoch
        row = {
            "fold": fold_number,
            "size": size,
            "stage": stage_name,
            "epoch": epoch + 1,
            "train_loss": epoch_loss,
            "lr": stlr_value(selected_lr, current_step, total_steps),
            **metrics,
            "train_seconds": epoch_seconds,
            "effective_images_per_second": (
                len(train_sampler) / max(epoch_seconds, 1e-9)
            ),
        }
        history.append(row)

        if float(metrics["dice"]) > best_score:
            best_score = float(metrics["dice"])
            best_metrics = metrics
            best_epoch = epoch + 1
            best_state = cpu_state_dict(model)

        should_save = (epoch + 1) % SAVE_LAST_EVERY == 0 or epoch + 1 == epochs
        if should_save:
            atomic_torch_save(
                {
                    "complete": False,
                    "contract": stage_contract,
                    "next_epoch": epoch + 1,
                    "selected_lr": selected_lr,
                    "lrf_lrs": lrf_lrs,
                    "lrf_losses": lrf_losses,
                    "best_score": best_score,
                    "best_metrics": best_metrics,
                    "best_epoch": best_epoch,
                    "best_model_state": best_state,
                    "history": history,
                    "model_state": cpu_state_dict(model),
                    "optimizer_state": tree_to_cpu(optimizer.state_dict()),
                },
                last_path,
            )

        print(
            f"[fold {fold_number} | {size}px | {stage_name}] "
            f"epoch {epoch + 1:02d}/{epochs} loss={epoch_loss:.4f} "
            f"dice={metrics['dice']:.4f} jac={metrics['jaccard']:.4f} "
            f"thr-jac={metrics['threshold_jaccard']:.4f} "
            f"lr={row['lr']:.2e} train={epoch_seconds:.1f}s "
            f"img/s={row['effective_images_per_second']:.1f}"
        )
        if epoch == start_epoch:
            print(met.short_metrics_report())

    atomic_torch_save(
        {
            "model_state": best_state,
            "meta": {
                "contract": stage_contract,
                "epoch": best_epoch,
                "metrics": best_metrics,
                "selected_lr": selected_lr,
            },
        },
        best_path,
    )
    load_model_state(model, best_state)
    result = {
        "fold": fold_number,
        "size": size,
        "stage": stage_name,
        "selected_lr": selected_lr,
        "best_epoch": best_epoch,
        "best_metrics": best_metrics,
        "history": history,
        "lrf_lrs": lrf_lrs,
        "lrf_losses": lrf_losses,
        "best_path": str(best_path),
    }
    # Marker hoàn tất nhỏ thay cho checkpoint optimizer lớn.
    atomic_torch_save(
        {"complete": True, "contract": stage_contract, "result": result},
        last_path,
    )
    write_rows(
        LOG_DIR / f"fold_{fold_number}_{size}px_{stage_name}.csv", history
    )
    del train_loader, holdout_loader, optimizer, best_state
    gc.collect()
    return result




In [ ]:
# ---------------------------------------------------------------------------
# Train folds và official evaluation
# ---------------------------------------------------------------------------

def make_eval_loader(
    records: Sequence[dict[str, str]],
    cache_spec: Mapping[str, Any],
    size: int,
    seed: int,
) -> DataLoader:
    base = CachedSegmentationDataset(records, cache_spec, train=False, seed=seed)
    dataset = PaddedEvalDataset(base, EVAL_BATCH_BY_SIZE[size])
    return make_cpu_loader(
        dataset,
        batch_size=EVAL_BATCH_BY_SIZE[size],
        workers=EVAL_NUM_WORKERS,
        seed=seed,
        sampler=None,
        persistent=True,
    )


def train_fold(
    fold_number: int,
    train_records: Sequence[dict[str, str]],
    holdout_records: Sequence[dict[str, str]],
    cache_specs: Mapping[int, Mapping[str, Any]],
    device: torch.device,
    input_sharding: Mapping[str, xs.ShardingSpec],
    mean: torch.Tensor,
    std: torch.Tensor,
    run_config: Mapping[str, Any],
) -> tuple[Path, dict[str, Any]]:
    final_path = CHECKPOINT_DIR / f"fold_{fold_number}" / "unet34_best.pt"
    if RESUME and final_path.is_file():
        payload = trusted_torch_load(final_path)
        meta = payload["meta"]
        if (
            meta.get("config") != dict(run_config)
            or meta.get("train_ids_sha256") != ids_digest(train_records)
            or meta.get("holdout_ids_sha256") != ids_digest(holdout_records)
        ):
            raise RuntimeError(f"Checkpoint final fold {fold_number} không tương thích")
        print(
            f"[resume] fold {fold_number} hoàn tất, "
            f"Dice={meta['holdout_metrics']['dice']:.4f}"
        )
        return final_path, meta

    fold_seed = SEED + fold_number - 1
    set_seed(fold_seed)
    model = UNet34(pretrained=PRETRAINED).to(device)
    stage_results: list[dict[str, Any]] = []
    for size_index, size in enumerate(SIZES):
        freeze_first_layer_group(model)
        stage_results.append(
            run_stage(
                model,
                train_records,
                holdout_records,
                cache_specs[size],
                size,
                FROZEN_EPOCHS,
                False,
                fold_number,
                "frozen",
                fold_seed + size_index * 1000,
                device,
                input_sharding,
                mean,
                std,
                run_config,
            )
        )
        unfreeze_with_frozen_bn(model)
        stage_results.append(
            run_stage(
                model,
                train_records,
                holdout_records,
                cache_specs[size],
                size,
                UNFROZEN_EPOCHS,
                True,
                fold_number,
                "unfrozen",
                fold_seed + size_index * 1000 + 500,
                device,
                input_sharding,
                mean,
                std,
                run_config,
            )
        )

    final_result = stage_results[-1]
    meta = {
        "fold": fold_number,
        "fold_seed": fold_seed,
        "config": dict(run_config),
        "train_count": len(train_records),
        "holdout_count": len(holdout_records),
        "train_ids_sha256": ids_digest(train_records),
        "holdout_ids_sha256": ids_digest(holdout_records),
        "holdout_metrics": final_result["best_metrics"],
        "stage_results": stage_results,
    }
    atomic_torch_save(
        {"model_state": cpu_state_dict(model), "meta": meta}, final_path
    )
    history = [row for result in stage_results for row in result["history"]]
    write_rows(LOG_DIR / f"fold_{fold_number}_history.csv", history)
    del model
    gc.collect()
    return final_path, meta


def load_runtime_model(path: Path, device: torch.device) -> UNet34:
    payload = trusted_torch_load(path)
    model = UNet34(pretrained=False)
    load_model_state(model, payload["model_state"])
    return model.to(device).eval()


def save_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    os.replace(temporary, path)


def validate_metric_implementation() -> None:
    target = torch.zeros(2, 1, 8, 8)
    target[0, :, :4] = 1
    target[1, :, :, :4] = 1
    probability = target.clone()
    probability[1] = 0
    probability[1, :, :, :2] = 1
    jaccard, _, _ = per_image_scores(probability, target)
    if not torch.allclose(jaccard, torch.tensor([1.0, 0.5])):
        raise AssertionError(f"Metric self-test thất bại: {jaccard.tolist()}")


def validate_cached_sample(
    records: Sequence[dict[str, str]], cache_spec: Mapping[str, Any], size: int
) -> None:
    dataset = CachedSegmentationDataset(
        records[:3], cache_spec, train=False, seed=SEED
    )
    for index in range(len(dataset)):
        first = dataset[index]
        second = dataset[index]
        image = first["image"]
        mask = first["mask"]
        if not torch.equal(image, second["image"]) or not torch.equal(
            mask, second["mask"]
        ):
            raise AssertionError("Clean transform phải deterministic")
        if image.shape != (3, size, size) or mask.shape != (1, size, size):
            raise AssertionError(f"Sai shape ảnh/mask: {image.shape}, {mask.shape}")
        unique = set(torch.unique(mask).tolist())
        if not unique.issubset({0.0, 1.0}):
            raise AssertionError(f"Mask không nhị phân: {unique}")
        foreground = float(mask.mean())
        if not 0.0 < foreground < 1.0:
            raise AssertionError(f"Mask bất thường, foreground={foreground}")


def version_pair(version: str) -> tuple[int, int]:
    numeric = version.split("+")[0].split("-")[0].split(".")
    if len(numeric) < 2 or not numeric[0].isdigit() or not numeric[1].isdigit():
        raise RuntimeError(f"Không đọc được major.minor từ version: {version!r}")
    return int(numeric[0]), int(numeric[1])


def main() -> None:
    cv2.setNumThreads(0)
    set_seed(SEED)
    for directory in (RUN_DIR, CHECKPOINT_DIR, LOG_DIR):
        directory.mkdir(parents=True, exist_ok=True)

    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(
            f"Chưa Attach dataset tinlngnguyn/skin-lesion-segmentation-hus: {DATA_ROOT}"
        )

    torch_version = str(torch.__version__)
    xla_version = str(getattr(torch_xla, "__version__", "unknown"))
    if version_pair(torch_version) != version_pair(xla_version):
        raise RuntimeError(
            f"torch {torch_version} và torch_xla {xla_version} không cùng major.minor. "
            "Dùng nguyên TPU image của Kaggle, không pip-upgrade riêng lẻ."
        )
    if version_pair(xla_version) < (2, 8):
        raise RuntimeError(
            f"Pipeline cần torch_xla >= 2.8 cho SPMD dict sharding; đang có {xla_version}."
        )
    print(
        {
            "python": sys.version.split()[0],
            "torch": torch_version,
            "torch_xla": xla_version,
            "torchvision": torchvision.__version__,
            "fast_dev_run": FAST_DEV_RUN,
        }
    )

    # Download đúng một lần trước khi khởi tạo XLA runtime.
    if PRETRAINED:
        print("Kiểm tra/tải ResNet34 ImageNet weights một lần...")
        models.ResNet34_Weights.IMAGENET1K_V1.get_state_dict(
            progress=True, check_hash=True
        )

    train_all = discover_records("train")
    official_val_all = discover_records("validation")
    test_all = discover_records("test")
    split_ids = {
        "train": {record["image_id"] for record in train_all},
        "validation": {record["image_id"] for record in official_val_all},
        "test": {record["image_id"] for record in test_all},
    }
    if not (
        split_ids["train"].isdisjoint(split_ids["validation"])
        and split_ids["train"].isdisjoint(split_ids["test"])
        and split_ids["validation"].isdisjoint(split_ids["test"])
    ):
        raise RuntimeError("Official train/validation/test bị trùng image id")

    if FAST_DEV_RUN:
        train_records = deterministic_subset(train_all, 256, SEED)
        official_val_records = deterministic_subset(official_val_all, 64, SEED + 1)
        test_records = deterministic_subset(test_all, 128, SEED + 2)
    else:
        train_records = train_all
        official_val_records = official_val_all
        test_records = test_all

    folds = make_folds(train_records)
    print(
        "counts:",
        len(train_records),
        len(official_val_records),
        len(test_records),
    )
    print("folds:", [(len(train), len(holdout)) for train, holdout in folds])

    cache_specs: dict[int, dict[str, Any]] = {}
    for size in SIZES:
        cache_records = list(train_records)
        if size == SIZES[-1]:
            cache_records.extend(official_val_records)
            cache_records.extend(test_records)
        cache_specs[size] = build_or_reuse_cache(cache_records, size)
        validate_cached_sample(train_records, cache_specs[size], size)
    cache_bytes = sum(
        Path(spec["image_path"]).stat().st_size
        + Path(spec["mask_path"]).stat().st_size
        for spec in cache_specs.values()
    )
    print(f"Cache uint8: {cache_bytes / 1024**3:.2f} GiB tại {DATA_CACHE_DIR}")
    validate_metric_implementation()

    # Compilation cache phải được khởi tạo trước phép tính XLA đầu tiên.
    XLA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    if hasattr(xr, "initialize_cache"):
        xr.initialize_cache(str(XLA_CACHE_DIR), readonly=False)
    xr.use_spmd()
    runtime_devices = int(xr.global_runtime_device_count())
    if runtime_devices != EXPECTED_TPU_DEVICES:
        raise RuntimeError(
            f"Cần TPU v5e-8 (8 chip), runtime chỉ thấy {runtime_devices} device. "
            "Chọn Accelerator = TPU v5e-8 rồi Restart Session."
        )
    for size, batch in {
        **TRAIN_BATCH_BY_SIZE,
        **{size + 10_000: value for size, value in EVAL_BATCH_BY_SIZE.items()},
    }.items():
        if batch % runtime_devices:
            raise ValueError(f"Global batch {batch} không chia hết cho 8 TPU chip")

    device_ids = np.arange(runtime_devices)
    mesh = xs.Mesh(device_ids, (runtime_devices,), ("data",))
    device = torch_xla.device()
    input_sharding = {
        "image": xs.ShardingSpec(mesh, ("data", None, None, None)),
        "mask": xs.ShardingSpec(mesh, ("data", None, None, None)),
        "valid": xs.ShardingSpec(mesh, ("data",)),
    }
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)

    run_config = {
        "protocol_version": PROTOCOL_VERSION,
        "model_revision": MODEL_REVISION,
        "cache_version": CACHE_VERSION,
        "seed": SEED,
        "n_folds": N_FOLDS,
        "sizes": list(SIZES),
        "frozen_epochs": FROZEN_EPOCHS,
        "unfrozen_epochs": UNFROZEN_EPOCHS,
        "train_batch_by_size": TRAIN_BATCH_BY_SIZE,
        "eval_batch_by_size": EVAL_BATCH_BY_SIZE,
        "local_train_batch_by_size": {
            size: batch // runtime_devices
            for size, batch in TRAIN_BATCH_BY_SIZE.items()
            if size in SIZES
        },
        "device": "TPU v5e-8",
        "execution": "PyTorch/XLA SPMD data parallel",
        "precision": "BF16 autocast; FP32 weights/optimizer/metrics",
        "optimizer": "Adam(capturable tensor LR, foreach=False)",
        "loss": "BCEWithLogitsLoss",
        "pretrained": PRETRAINED,
        "weight_decay": WEIGHT_DECAY,
        "grad_clip_norm": GRAD_CLIP_NORM,
        "lr_finder": {
            "min": LR_MIN,
            "max": LR_MAX,
            "iterations": LRF_ITERS,
            "beta": LRF_BETA,
            "selection_factor": LR_SELECTION_FACTOR,
            "max_lr_frozen": MAX_LR_FROZEN,
            "max_lr_unfrozen": MAX_LR_UNFROZEN,
        },
        "stlr": {"cut_frac": CUT_FRAC, "ratio": STLR_RATIO},
        "prediction_threshold": PRED_THRESHOLD,
        "jaccard_cut": JACCARD_CUT,
        "encoder_weights": "ResNet34_Weights.IMAGENET1K_V1",
        "augmentation": "train-only: D4+Affine44+zoom1.05+RBC0.2",
        "selection_metric": "fold holdout dice",
        "official_validation_role": "final reporting only",
        "train_padding": "shuffle then repeat to static global batch",
        "eval_padding": "static batch with valid mask",
        "train_workers": TRAIN_NUM_WORKERS,
        "eval_workers": EVAL_NUM_WORKERS,
        "cache": "uint8 np.memmap in /tmp",
        "fast_dev_run": FAST_DEV_RUN,
    }
    save_json(RUN_DIR / "config.json", run_config)
    print(
        {
            "device": str(device),
            "runtime_devices": runtime_devices,
            "global_train_batch": {
                size: TRAIN_BATCH_BY_SIZE[size] for size in SIZES
            },
            "per_chip_train_batch": run_config["local_train_batch_by_size"],
            "output": str(RUN_DIR),
        }
    )

    final_paths: list[Path] = []
    fold_meta: list[dict[str, Any]] = []
    training_started = time.time()
    for fold_number, (fold_train, fold_holdout) in enumerate(folds, start=1):
        print(
            f"\n===== FOLD {fold_number}/{N_FOLDS}: "
            f"train={len(fold_train)}, holdout={len(fold_holdout)} ====="
        )
        final_path, meta = train_fold(
            fold_number,
            fold_train,
            fold_holdout,
            cache_specs,
            device,
            input_sharding,
            mean,
            std,
            run_config,
        )
        final_paths.append(final_path)
        fold_meta.append(meta)

    fold_rows = [
        {"fold": meta["fold"], **meta["holdout_metrics"]} for meta in fold_meta
    ]
    write_rows(LOG_DIR / "fold_holdout_metrics.csv", fold_rows)
    print(f"Train/resume xong sau {(time.time() - training_started) / 3600:.2f} giờ")

    # Chỉ sau khi khóa BestDice của cả 3 fold mới đụng official validation/test.
    runtime_models = [load_runtime_model(path, device) for path in final_paths]
    final_size = SIZES[-1]
    validation_rows: list[dict[str, Any]] = []
    validation_loader = make_eval_loader(
        official_val_records, cache_specs[final_size], final_size, SEED
    )
    for index, model in enumerate(runtime_models, start=1):
        metrics = evaluate(
            [model],
            validation_loader,
            device,
            input_sharding,
            mean,
            std,
            include_hd95=True,
        )
        validation_rows.append({"model": f"BestDice fold {index}", **metrics})
    validation_ensemble = evaluate(
        runtime_models,
        validation_loader,
        device,
        input_sharding,
        mean,
        std,
        include_hd95=True,
    )
    validation_rows.append(
        {"model": "Equal-weight ensemble", **validation_ensemble}
    )
    del validation_loader

    selected_single_index = int(
        np.argmax([meta["holdout_metrics"]["dice"] for meta in fold_meta])
    )
    selected_single_fold = selected_single_index + 1
    test_loader = make_eval_loader(
        test_records, cache_specs[final_size], final_size, SEED
    )
    test_single = evaluate(
        [runtime_models[selected_single_index]],
        test_loader,
        device,
        input_sharding,
        mean,
        std,
        include_hd95=True,
    )
    test_ensemble = evaluate(
        runtime_models,
        test_loader,
        device,
        input_sharding,
        mean,
        std,
        include_hd95=True,
    )
    test_rows = [
        {"model": f"Single BestDice fold {selected_single_fold}", **test_single},
        {"model": "Equal-weight ensemble", **test_ensemble},
    ]
    write_rows(LOG_DIR / "official_validation_metrics.csv", validation_rows)
    write_rows(LOG_DIR / "official_test_metrics.csv", test_rows)
    summary = {
        "config": run_config,
        "sample_counts": {
            "train": len(train_records),
            "validation": len(official_val_records),
            "test": len(test_records),
        },
        "fold_holdout": fold_rows,
        "selected_single_fold_by_cv_dice": selected_single_fold,
        "official_validation": validation_rows,
        "official_test": test_rows,
    }
    save_json(RUN_DIR / "summary.json", summary)
    print("\nOfficial validation:", json.dumps(validation_rows, indent=2))
    print("Official test:", json.dumps(test_rows, indent=2))
    print("Summary:", RUN_DIR / "summary.json")


if __name__ == "__main__":
    main()
